# LeetCode 2300 — Successful Pairs of Spells and Potions bằng Python

Link đề: https://leetcode.com/problems/successful-pairs-of-spells-and-potions/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def successfulPairs(self, spells: List[int], potions: List[int], success: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Biến điều kiện tích `spell · potion ≥ success` thành **ngưỡng một chiều** trên `potion`: `potion ≥ ⌈success / spell⌉`.
2. Sắp xếp `potions` một lần rồi trả lời mỗi `spell` bằng **tìm nhị phân** (`bisect_left`) — O((n + m) log m).
3. Tính trần **bằng số nguyên** `(a + b − 1) // b` (hoặc `−(−a // b)`) thay vì số thực, và hiểu khi nào `success / v` kiểu float bắt đầu sai.
4. Cách **offline** hai con trỏ: sắp xếp cả `spells` (giữ chỉ số gốc) để ngưỡng di chuyển đơn điệu.
5. Cách **đếm phân phối** + tổng hậu tố khi giá trị ≤ 10⁵: trả lời mỗi truy vấn O(1).

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên dương `spells` (độ dài `n`) và `potions` (độ dài `m`), và số nguyên `success`. Một cặp (bùa, thuốc) **thành công** nếu tích sức mạnh của chúng **≥ `success`**. Trả về mảng `pairs` độ dài `n`, `pairs[i]` = số lọ thuốc tạo cặp thành công với bùa `i`.

**Ví dụ 1:** `spells = [5,1,3]`, `potions = [1,2,3,4,5]`, `success = 7` → `[4,0,3]`.

**Ví dụ 2:** `spells = [3,1,2]`, `potions = [8,5,8]`, `success = 16` → `[2,0,2]`.

**Ràng buộc:** `1 ≤ n, m ≤ 10⁵`; `1 ≤ spells[i], potions[j] ≤ 10⁵`; `1 ≤ success ≤ 10¹⁰`.

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
from itertools import accumulate
import random, time

examples = [([5, 1, 3], [1, 2, 3, 4, 5], 7, [4, 0, 3]), ([3, 1, 2], [8, 5, 8], 16, [2, 0, 2])]
for sp, po, s, exp in examples:
    print(f"spells={sp} potions={po} success={s} -> kỳ vọng {exp}")

spells=[5, 1, 3] potions=[1, 2, 3, 4, 5] success=7 -> kỳ vọng [4, 0, 3]
spells=[3, 1, 2] potions=[8, 5, 8] success=16 -> kỳ vọng [2, 0, 2]


## 3. Kiến thức nền

### 3.1 Từ điều kiện tích sang ngưỡng

Với `v > 0`: `v · p ≥ s` ⇔ `p ≥ s / v` ⇔ (vì `p` nguyên) `p ≥ ⌈s / v⌉`. Ngưỡng `need = ⌈s / v⌉` chỉ phụ thuộc vào bùa.

In [2]:
s = 7
for v in [5, 1, 3]:
    need = (s + v - 1) // v
    print(f"v={v}: need = ceil({s}/{v}) = {need}; kiểm tra: {v}*{need} = {v * need} >= {s}, {v}*{need - 1} = {v * (need - 1)} < {s}")

v=5: need = ceil(7/5) = 2; kiểm tra: 5*2 = 10 >= 7, 5*1 = 5 < 7
v=1: need = ceil(7/1) = 7; kiểm tra: 1*7 = 7 >= 7, 1*6 = 6 < 7
v=3: need = ceil(7/3) = 3; kiểm tra: 3*3 = 9 >= 7, 3*2 = 6 < 7


### 3.2 Chia lấy trần bằng số nguyên

`⌈a / b⌉ = (a + b − 1) // b = −(−a // b)` với `a ≥ 0, b > 0`. Không dùng `math.ceil(a / b)` khi `a` lớn (qua float, mất chính xác khi vượt 2⁵³).

In [3]:
import math
a, b = 10**17 + 1, 1
print((a + b - 1) // b, -(-a // b), math.ceil(a / b), "<- float làm tròn sai")

100000000000000001 100000000000000001 100000000000000000 <- float làm tròn sai


### 3.3 `bisect_left` / `bisect_right`

Trên mảng tăng dần `A`: `bisect_left(A, t)` = số phần tử `< t` = chỉ số phần tử `≥ t` đầu tiên. Số phần tử `≥ t` là `len(A) − bisect_left(A, t)`.

In [4]:
A = [1, 2, 3, 3, 3, 5]
for t in [0, 3, 4, 6]:
    print(f"t={t}: bisect_left={bisect_left(A, t)}, bisect_right={bisect_right(A, t)}, số phần tử >= t = {len(A) - bisect_left(A, t)}")

t=0: bisect_left=0, bisect_right=0, số phần tử >= t = 6
t=3: bisect_left=2, bisect_right=5, số phần tử >= t = 4
t=4: bisect_left=5, bisect_right=5, số phần tử >= t = 1
t=6: bisect_left=6, bisect_right=6, số phần tử >= t = 0


### 3.4 Tổng hậu tố trên bảng đếm

`cnt[x]` = số lọ có sức mạnh `x`; `suf[t] = Σ_{x ≥ t} cnt[x]` ⇒ "số lọ ≥ t" trả lời O(1).

In [5]:
po = [1, 2, 3, 4, 5]; V = 6
cnt = [0] * (V + 1)
for p in po: cnt[p] += 1
suf = list(accumulate(reversed(cnt)))[::-1] + [0]
print("cnt:", cnt, "| suf:", suf, "| số lọ >= 3:", suf[3])

cnt: [0, 1, 1, 1, 1, 1, 0] | suf: [5, 5, 4, 3, 2, 1, 0, 0] | số lọ >= 3: 3


### 3.5 So sánh `int` với `float` trong Python là **chính xác**

Python không ép số nguyên thành float khi so sánh: `2**53 + 1 > float(2**53)` là `True` dù `float(2**53 + 1) == float(2**53)`. Vì vậy `bisect_left(P, success / v)` chỉ chịu sai số ở **một** chỗ: phép chia `success / v`.

In [6]:
x = 2**53 + 1
print(x > float(2**53), float(x) == float(2**53), 7 / 5, 7 / 5 > 1, bisect_left([1, 2, 3], 7 / 5))

True True 1.4 True 1


## 4. Ý tưởng nền tảng / trực giác

Cố định bùa `v`. Tích `v · p` **tăng** theo `p` ⇒ nếu sắp xếp `potions` tăng dần thì các lọ thành công tạo thành một **hậu tố**. Chỉ cần tìm điểm bắt đầu của hậu tố đó — tìm nhị phân.

Tính tay ví dụ 2: `potions` sắp xếp `[5, 8, 8]`, `success = 16`.

| bùa v | need = ⌈16 / v⌉ | chỉ số đầu tiên ≥ need | số lọ |
|---|---|---|---|
| 3 | 6 | 1 | 3 − 1 = **2** |
| 1 | 16 | 3 | **0** |
| 2 | 8 | 1 | **2** |

⇒ `[2, 0, 2]` ✔.

Nhận xét thêm: `need` **giảm** khi `v` tăng ⇒ nếu xử lý các bùa theo thứ tự `v` tăng dần, điểm bắt đầu hậu tố chỉ di chuyển **sang trái** ⇒ hai con trỏ (cách offline).

In [7]:
sp, po, s, _ = examples[1]
P = sorted(po)
for v in sp:
    need = -(-s // v); k = bisect_left(P, need)
    print(f"v={v}: need={need}, P={P}, bắt đầu từ chỉ số {k} -> {len(P) - k} lọ: {P[k:]}")

v=3: need=6, P=[5, 8, 8], bắt đầu từ chỉ số 1 -> 2 lọ: [8, 8]
v=1: need=16, P=[5, 8, 8], bắt đầu từ chỉ số 3 -> 0 lọ: []
v=2: need=8, P=[5, 8, 8], bắt đầu từ chỉ số 1 -> 2 lọ: [8, 8]


### 4.1 Ma trận thành công có dạng **bậc thang**

Sắp xếp cả bùa (hàng) và lọ (cột) tăng dần rồi đánh dấu `#` các cặp thành công: mỗi hàng là một **hậu tố**, và hậu tố dài dần khi đi xuống. Tìm nhị phân là tìm điểm bắt đầu hậu tố trên từng hàng; hai con trỏ là đi dọc "đường biên bậc thang" từ trên xuống, chỉ sang trái.

In [8]:
sp, po, s = [5, 1, 3, 2, 4], [1, 2, 3, 4, 5], 7
S, P = sorted(sp), sorted(po)
print("v \\ p " + " ".join(f"{p:>2}" for p in P))
for v in S:
    print(f"{v:>6} " + " ".join(" #" if v * p >= s else " ." for p in P), "  ->", sum(v * p >= s for p in P))

v \ p  1  2  3  4  5
     1  .  .  .  .  .   -> 0
     2  .  .  .  #  #   -> 2
     3  .  .  #  #  #   -> 3
     4  .  #  #  #  #   -> 4
     5  .  #  #  #  #   -> 4


Đường biên đi từ phải sang trái khi `v` tăng — đó chính là tính đơn điệu mà cả `bisect` và hai con trỏ khai thác. Tổng số bước của con trỏ ≤ số cột ⇒ O(m) sau khi sắp xếp.

## 5. Cách 1 — Brute Force / baseline

### 5.1 Thử mọi cặp — O(n · m)

In [9]:
def brute(spells, potions, success):
    return [sum(1 for p in potions if v * p >= success) for v in spells]

for sp, po, s, exp in examples:
    print(brute(sp, po, s), "OK" if brute(sp, po, s) == exp else "SAI")

[4, 0, 3] OK
[2, 0, 2] OK


`n · m = 10¹⁰` phép nhân — quá chậm.

### 5.2 Baseline 2 — Sắp xếp `potions`, quét tuyến tính từ phải cho mỗi bùa — O(n · m) worst

Dừng sớm khi gặp lọ không đủ — nhanh khi đáp án nhỏ, nhưng worst case (mọi lọ đều thành công) vẫn O(n · m).

In [10]:
def sorted_scan(spells, potions, success):
    P = sorted(potions); out = []
    for v in spells:
        c = 0
        for p in reversed(P):
            if v * p < success: break
            c += 1
        out.append(c)
    return out

for sp, po, s, exp in examples:
    print(sorted_scan(sp, po, s), "OK" if sorted_scan(sp, po, s) == exp else "SAI")

[4, 0, 3] OK
[2, 0, 2] OK


**Nút thắt:** việc tìm "ranh giới thành công / thất bại" trên mảng đã sắp làm tuần tự. Ranh giới đó là điểm **đổi tính chất đơn điệu** ⇒ tìm nhị phân O(log m).

## 6. Ý tưởng thuật toán / bước đột phá

```
P = sorted(potions)
for v in spells:
    need = ceil(success / v)                 # bằng số nguyên
    answer.append(m − bisect_left(P, need))
```

- Sắp xếp một lần: O(m log m).
- Mỗi bùa: O(log m).
- Tổng: O((n + m) log m), bộ nhớ O(m).

**Cải tiến khi giá trị nhỏ:** `potions[j] ≤ 10⁵` ⇒ bảng đếm + tổng hậu tố O(m + V), mỗi bùa O(1); nếu `need > max(potions)` thì đáp án 0 (nhớ chặn chỉ số — `need` có thể tới `10¹⁰`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + `bisect` với float (cách doocs)

In [11]:
def bisect_float(spells, potions, success):
    P = sorted(potions); m = len(P)
    return [m - bisect_left(P, success / v) for v in spells]

for sp, po, s, exp in examples:
    print(bisect_float(sp, po, s), "OK" if bisect_float(sp, po, s) == exp else "SAI")

[4, 0, 3] OK
[2, 0, 2] OK


`bisect_left(P, success / v)` so sánh số nguyên với float — Python so sánh **chính xác** int với float, nên chỉ cần `success / v` đủ gần giá trị thật. Với giới hạn đề, sai số của phép chia float (tương đối ~`10⁻¹⁶`) nhỏ hơn rất nhiều khoảng cách tới số nguyên gần nhất khi ngưỡng ≤ 10⁵ — an toàn. Mục 15.2 cho thấy khi giá trị lớn tới `10¹⁸` thì float sai.

### 7.2 Cách 3 — Sắp xếp + `bisect` với trần số nguyên (khuyến nghị)

In [12]:
def bisect_int(spells, potions, success):
    P = sorted(potions); m = len(P)
    return [m - bisect_left(P, (success + v - 1) // v) for v in spells]

for sp, po, s, exp in examples:
    print(bisect_int(sp, po, s), "OK" if bisect_int(sp, po, s) == exp else "SAI")

[4, 0, 3] OK
[2, 0, 2] OK


### 7.3 Cách 4 — Offline: sắp xếp cả bùa, hai con trỏ — O(n log n + m log m)

In [13]:
def two_pointers(spells, potions, success):
    P = sorted(potions); m = len(P)
    order = sorted(range(len(spells)), key=spells.__getitem__)    # bùa yếu -> mạnh
    out = [0] * len(spells); k = m                                  # k: chỉ số đầu của hậu tố thành công
    for i in order:
        v = spells[i]
        while k > 0 and v * P[k - 1] >= success:                    # bùa mạnh hơn => hậu tố mở rộng sang trái
            k -= 1
        out[i] = m - k
    return out

for sp, po, s, exp in examples:
    print(two_pointers(sp, po, s), "OK" if two_pointers(sp, po, s) == exp else "SAI")

[4, 0, 3] OK
[2, 0, 2] OK


Con trỏ `k` chỉ giảm ⇒ tổng số bước O(m). Không cần chia — chỉ nhân, nên cũng không lo trần/làm tròn.

### 7.4 Cách 5 — Đếm phân phối + tổng hậu tố — O(n + m + V)

In [14]:
def counting(spells, potions, success):
    V = max(potions)
    suf = [0] * (V + 2)
    for p in potions: suf[p] += 1
    for x in range(V - 1, 0, -1): suf[x] += suf[x + 1]
    out = []
    for v in spells:
        need = (success + v - 1) // v
        out.append(suf[need] if need <= V else 0)
    return out

for sp, po, s, exp in examples:
    print(counting(sp, po, s), "OK" if counting(sp, po, s) == exp else "SAI")

[4, 0, 3] OK
[2, 0, 2] OK


`need` luôn ≥ 1 (vì `success ≥ 1`), nên `suf[need]` với `need ≤ V` là hợp lệ; `need > V` ⇒ không lọ nào đủ.

### 7.5 Cách 6 — Nhớ kết quả theo giá trị bùa (bùa trùng nhau)

Nhiều bùa cùng giá trị ⇒ tính một lần. Kết hợp với bất kỳ cách nào ở trên.

In [15]:
def memo_by_value(spells, potions, success):
    P = sorted(potions); m = len(P); cache = {}
    for v in set(spells):
        cache[v] = m - bisect_left(P, (success + v - 1) // v)
    return [cache[v] for v in spells]

for sp, po, s, exp in examples:
    print(memo_by_value(sp, po, s), "OK" if memo_by_value(sp, po, s) == exp else "SAI")

[4, 0, 3] OK
[2, 0, 2] OK


### 7.6 Cách 7 — Đảo vai: mỗi lọ "cộng 1" cho mọi bùa đủ mạnh (mảng hiệu trên bùa đã sắp)

Với lọ `p`, các bùa thành công là `v ≥ ⌈s / p⌉` — một **hậu tố** của `spells` đã sắp. Cộng 1 cho cả hậu tố bằng mảng hiệu (difference array), cuối cùng lấy tổng tiền tố và trả về theo chỉ số gốc. Cách này hữu ích khi `m ≪ n` hoặc khi cần đếm theo phía lọ thuốc.

In [16]:
def swap_roles(spells, potions, success):
    n = len(spells); order = sorted(range(n), key=spells.__getitem__); S = [spells[i] for i in order]
    diff = [0] * (n + 1)
    for p in potions:
        k = bisect_left(S, (success + p - 1) // p)        # bùa đầu tiên đủ mạnh với lọ p
        diff[k] += 1                                      # +1 cho S[k:], tức diff[k] += 1, diff[n] -= 1
    out = [0] * n; run = 0
    for r in range(n):
        run += diff[r]; out[order[r]] = run
    return out

for sp, po, s, exp in examples:
    print(swap_roles(sp, po, s), "OK" if swap_roles(sp, po, s) == exp else "SAI")

[4, 0, 3] OK
[2, 0, 2] OK


Vì mọi hậu tố đều kết thúc ở cuối mảng, `diff[n] −= 1` không cần thiết (không bao giờ được đọc). Độ phức tạp O(n log n + m log n).

## 8. TRACE từng bước

### 8.1 Trace tìm nhị phân trên ví dụ 1 (in từng bước `lo/hi`)

In [17]:
def lower_bound_trace(A, t):
    lo, hi = 0, len(A); steps = []
    while lo < hi:
        mid = (lo + hi) // 2
        steps.append(f"[{lo},{hi}) mid={mid} A[mid]={A[mid]} {'<' if A[mid] < t else '>='} {t}")
        if A[mid] < t: lo = mid + 1
        else: hi = mid
    return lo, steps

sp, po, s, _ = examples[0]; P = sorted(po)
for v in sp:
    need = -(-s // v); k, steps = lower_bound_trace(P, need)
    print(f"bùa {v}: need = {need}"); [print("    ", st) for st in steps]; print(f"    -> chỉ số {k}, đáp án {len(P) - k}")

bùa 5: need = 2
     [0,5) mid=2 A[mid]=3 >= 2
     [0,2) mid=1 A[mid]=2 >= 2
     [0,1) mid=0 A[mid]=1 < 2
    -> chỉ số 1, đáp án 4
bùa 1: need = 7
     [0,5) mid=2 A[mid]=3 < 7
     [3,5) mid=4 A[mid]=5 < 7
    -> chỉ số 5, đáp án 0
bùa 3: need = 3
     [0,5) mid=2 A[mid]=3 >= 3
     [0,2) mid=1 A[mid]=2 < 3
    -> chỉ số 2, đáp án 3


### 8.2 Trace hai con trỏ (bùa theo thứ tự tăng dần)

In [18]:
sp, po, s, _ = examples[0]; P = sorted(po); m = len(P); k = m
for i in sorted(range(len(sp)), key=sp.__getitem__):
    v = sp[i]; moved = []
    while k > 0 and v * P[k - 1] >= s:
        k -= 1; moved.append(P[k])
    print(f"bùa #{i} (v={v}): con trỏ kéo qua {moved} -> k={k}, đáp án {m - k}")

bùa #1 (v=1): con trỏ kéo qua [] -> k=5, đáp án 0
bùa #2 (v=3): con trỏ kéo qua [5, 4, 3] -> k=2, đáp án 3
bùa #0 (v=5): con trỏ kéo qua [2] -> k=1, đáp án 4


Bùa 1 (yếu nhất): `1 · 5 = 5 < 7` ⇒ con trỏ đứng yên, 0 lọ. Bùa 3: kéo qua `5, 4, 3` (`3·3 = 9 ≥ 7`, `3·2 = 6 < 7`) ⇒ 3 lọ. Bùa 5: kéo tiếp qua `2` (`5·2 = 10`), dừng ở `1` (`5 < 7`) ⇒ 4 lọ.

## 9. Code nộp LeetCode

In [19]:
from typing import List
from bisect import bisect_left

class Solution:
    def successfulPairs(self, spells: List[int], potions: List[int], success: int) -> List[int]:
        potions = sorted(potions)
        m = len(potions)
        # lọ p thành công với bùa v  <=>  p >= ceil(success / v)
        return [m - bisect_left(potions, (success + v - 1) // v) for v in spells]

In [20]:
sol = Solution()
for sp, po, s, exp in examples:
    got = sol.successfulPairs(sp, po, s)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

[4, 0, 3] | kỳ vọng [4, 0, 3] | OK
[2, 0, 2] | kỳ vọng [2, 0, 2] | OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (ngưỡng).** Với số nguyên dương `v, p` và `s ≥ 1`: `v · p ≥ s` ⇔ `p ≥ ⌈s / v⌉`.
*Chứng minh:* `v · p ≥ s` ⇔ `p ≥ s / v` (chia cho `v > 0`). Vì `p` nguyên, `p ≥ s / v` ⇔ `p ≥ ⌈s / v⌉` (số nguyên nhỏ nhất ≥ `s / v`).

**Bổ đề 2 (trần nguyên).** Với `a ≥ 0`, `b > 0`: `(a + b − 1) // b = ⌈a / b⌉`.
*Chứng minh:* viết `a = qb + r`, `0 ≤ r < b`. Nếu `r = 0`: `(qb + b − 1) // b = q`. Nếu `r ≥ 1`: `(qb + r + b − 1) // b = q + 1` vì `b ≤ r + b − 1 < 2b`. Cả hai trường hợp đều bằng `⌈a / b⌉`.

**Bổ đề 3 (đếm bằng `bisect_left`).** Trên mảng tăng dần `P`, `bisect_left(P, t)` là số phần tử `< t`; vì mảng tăng, các phần tử `≥ t` là đúng `m − bisect_left(P, t)` phần tử cuối.

Ghép lại: `pairs[i] = #{j : potions[j] ≥ ⌈s / spells[i]⌉} = m − bisect_left(P, ⌈s / spells[i]⌉)`. Sắp xếp không đổi đa tập `potions` nên số đếm không đổi. ∎

**Đảo vai (cách 7).** Lọ `p` thành công với bùa `v` ⇔ `v ≥ ⌈s / p⌉` (đối xứng của bổ đề 1). Trên `spells` đã sắp, các bùa đó là hậu tố bắt đầu tại `k = bisect_left(S, ⌈s/p⌉)`. `diff[k] += 1` rồi lấy tổng tiền tố ⇒ vị trí `r` nhận đúng số lọ có `k ≤ r`, tức số lọ thành công với bùa `S[r]`.

**Hai con trỏ.** Với `v₁ ≤ v₂`, tập lọ thành công của `v₁` là tập con của tập của `v₂` (`v₁ p ≥ s ⇒ v₂ p ≥ s`), và đều là hậu tố của `P` ⇒ điểm bắt đầu hậu tố không tăng khi `v` tăng ⇒ con trỏ chỉ giảm, mỗi lần dừng đúng tại ranh giới.

## 11. Phân tích Time / Space Complexity

`n, m ≤ 10⁵`, `V = max(potions) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | O(n · m) | O(1) |
| 1b. Quét từ phải | O(n · m) worst | O(m) |
| 2–3. Sắp xếp + `bisect` | O((n + m) log m) | O(m) |
| 4. Hai con trỏ | O(n log n + m log m) | O(n + m) |
| 5. Đếm + hậu tố | O(n + m + V) | O(V) |
| 6. Nhớ theo giá trị | O(m log m + u log m) | O(m + u) |
| 7. Đảo vai + mảng hiệu | O(n log n + m log n) | O(n) |

In [21]:
rng = random.Random(2300)
sp = [rng.randint(1, 10**5) for _ in range(10**5)]; po = [rng.randint(1, 10**5) for _ in range(10**5)]
for s in [1, 10**5, 10**9, 10**10]:
    res = []
    for name, f in [("bisect_float", bisect_float), ("bisect_int", bisect_int), ("two_pointers", two_pointers),
                    ("counting", counting), ("memo_by_value", memo_by_value), ("swap_roles", swap_roles),
                    ("Solution", Solution().successfulPairs)]:
        t0 = time.perf_counter(); res.append(f(sp, po, s)); dt = time.perf_counter() - t0
        print(f"success={s:<12} {name:<14} {dt*1000:7.1f} ms  tổng = {sum(res[-1])}")
    assert all(r == res[0] for r in res)

success=1            bisect_float      40.8 ms  tổng = 10000000000


success=1            bisect_int        27.3 ms  tổng = 10000000000


success=1            two_pointers      52.4 ms  tổng = 10000000000


success=1            counting          14.0 ms  tổng = 10000000000


success=1            memo_by_value     37.5 ms  tổng = 10000000000
success=1            swap_roles        66.7 ms  tổng = 10000000000


success=1            Solution          43.4 ms  tổng = 10000000000


... (7 outputs left out)

success=1000000000   swap_roles       106.9 ms  tổng = 6689805874
success=1000000000   Solution          65.3 ms  tổng = 6689805874


success=10000000000  bisect_float      57.5 ms  tổng = 2
success=10000000000  bisect_int        42.5 ms  tổng = 2


success=10000000000  two_pointers      65.9 ms  tổng = 2
success=10000000000  counting          17.1 ms  tổng = 2
success=10000000000  memo_by_value     45.6 ms  tổng = 2


success=10000000000  swap_roles        56.1 ms  tổng = 2
success=10000000000  Solution          31.1 ms  tổng = 2


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [22]:
ALL = {"brute": brute, "sorted_scan": sorted_scan, "bisect_float": bisect_float, "bisect_int": bisect_int,
       "two_pointers": two_pointers, "counting": counting, "memo_by_value": memo_by_value, "swap_roles": swap_roles,
       "Solution": Solution().successfulPairs}
edge = examples + [
    ([1], [1], 1, [1]),                                     # tích đúng bằng success
    ([1], [1], 2, [0]),
    ([10**5], [10**5], 10**10, [1]),                        # cực đại: 10^5 * 10^5 = 10^10
    ([10**5 - 1], [10**5], 10**10, [0]),
    ([2, 2, 2], [3, 3, 3, 3], 6, [4, 4, 4]),               # bùa trùng, thuốc trùng, đúng biên
    ([4], [1, 2, 3], 8, [2]),                               # need = 2: lọ 2 (tích đúng bằng 8) và lọ 3
    ([7], [1, 2, 3], 1, [3]),                               # success = 1: mọi cặp
]
for sp, po, s, exp in edge:
    ref = brute(sp, po, s)
    for name, f in ALL.items():
        assert f(sp, list(po), s) == ref, (name, sp, po, s)
    print(f"spells={sp!s:<10} potions={po!s:<14} success={s:<12} -> {ref} | ghi trong bảng: {exp} | khớp: {ref == exp}")

spells=[5, 1, 3]  potions=[1, 2, 3, 4, 5] success=7            -> [4, 0, 3] | ghi trong bảng: [4, 0, 3] | khớp: True
spells=[3, 1, 2]  potions=[8, 5, 8]      success=16           -> [2, 0, 2] | ghi trong bảng: [2, 0, 2] | khớp: True
spells=[1]        potions=[1]            success=1            -> [1] | ghi trong bảng: [1] | khớp: True
spells=[1]        potions=[1]            success=2            -> [0] | ghi trong bảng: [0] | khớp: True
spells=[100000]   potions=[100000]       success=10000000000  -> [1] | ghi trong bảng: [1] | khớp: True
spells=[99999]    potions=[100000]       success=10000000000  -> [0] | ghi trong bảng: [0] | khớp: True
spells=[2, 2, 2]  potions=[3, 3, 3, 3]   success=6            -> [4, 4, 4] | ghi trong bảng: [4, 4, 4] | khớp: True
spells=[4]        potions=[1, 2, 3]      success=8            -> [2] | ghi trong bảng: [2] | khớp: True
spells=[7]        potions=[1, 2, 3]      success=1            -> [3] | ghi trong bảng: [3] | khớp: True


Case `[4], [1, 2, 3], 8`: `need = ⌈8/4⌉ = 2` ⇒ các lọ `2, 3` đều thành công (`4·2 = 8 ≥ 8`, `4·3 = 12`) ⇒ **2**. Nếu quên rằng điều kiện là **≥** (bằng cũng tính) sẽ ghi nhầm 1 — cột `khớp:` trong cell trên là lưới an toàn cho loại nhầm lẫn đó.

### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [23]:
rng = random.Random(42)
for _ in range(2000):
    sp = [rng.randint(1, 30) for _ in range(rng.randint(1, 8))]; po = [rng.randint(1, 30) for _ in range(rng.randint(1, 8))]
    s = rng.randint(1, 900)
    ref = brute(sp, po, s)
    for name, f in ALL.items():
        assert f(sp, list(po), s) == ref, (name, sp, po, s)
print("2000 bộ ngẫu nhiên OK")

2000 bộ ngẫu nhiên OK


### 12.3 Tính chất: bùa mạnh hơn ⇒ không ít cặp hơn; tăng `success` ⇒ không nhiều cặp hơn; `0 ≤ pairs[i] ≤ m`

In [24]:
rng = random.Random(7)
for _ in range(500):
    sp = [rng.randint(1, 100) for _ in range(rng.randint(1, 20))]; po = [rng.randint(1, 100) for _ in range(rng.randint(1, 20))]
    s = rng.randint(1, 10**4)
    r = Solution().successfulPairs(sp, po, s)
    assert all(0 <= x <= len(po) for x in r)
    for i in range(len(sp)):
        for j in range(len(sp)):
            if sp[i] <= sp[j]: assert r[i] <= r[j]
    r2 = Solution().successfulPairs(sp, po, s + rng.randint(0, 500))
    assert all(a >= b for a, b in zip(r, r2))
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: kiểm tra biên `v · p = success` với giá trị lớn

In [25]:
rng = random.Random(124)
for _ in range(2000):
    v = rng.randint(1, 10**5); p = rng.randint(1, 10**5); s = v * p
    assert Solution().successfulPairs([v], [p - 1, p, p + 1], s) == [2]           # p và p+1
    assert Solution().successfulPairs([v], [p - 1, p, p + 1], s + 1) == [1 if v * (p + 1) >= s + 1 else 0]
print("biên tích = success OK")

biên tích = success OK


### 12.5 Nhiều bùa trùng giá trị, kích thước tối đa — các cách cho cùng kết quả

In [26]:
rng = random.Random(125)
sp = [rng.choice([1, 7, 99999, 100000]) for _ in range(10**5)]; po = [rng.randint(1, 10**5) for _ in range(10**5)]
s = 7 * 10**9
base = Solution().successfulPairs(sp, po, s)
assert memo_by_value(sp, po, s) == counting(sp, po, s) == swap_roles(sp, po, s) == two_pointers(sp, po, s) == base
print("4 giá trị bùa khác nhau, số cặp theo giá trị:", {v: c for v, c in zip(sp, base)})

4 giá trị bùa khác nhau, số cặp theo giá trị: {7: 0, 99999: 30190, 1: 0, 100000: 30191}


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng chia **sàn** `success // v` làm ngưỡng

In [27]:
def wrong_floor(spells, potions, success):
    P = sorted(potions); m = len(P)
    return [m - bisect_left(P, success // v) for v in spells]       # SAI: cần trần

for sp, po, s, _ in examples:
    print("bản sai :", wrong_floor(sp, po, s), "| bản đúng:", Solution().successfulPairs(sp, po, s))

bản sai : [5, 0, 4] | bản đúng: [4, 0, 3]
bản sai : [3, 0, 2] | bản đúng: [2, 0, 2]


**Vì sao sai:** ví dụ 1, bùa 5: `7 // 5 = 1` ⇒ đếm cả lọ 1 (`5·1 = 5 < 7`). Khi `v` không chia hết `success`, sàn nhỏ hơn ngưỡng thật 1 đơn vị. Ví dụ 2 **trùng hợp** đúng ở bùa 2 (`16 / 2 = 8` chia hết) nhưng sai ở bùa 3 (`16 // 3 = 5` đếm cả lọ 5: `15 < 16`). **Sửa:** `(success + v − 1) // v`.

### Lỗi 2 — Dùng `bisect_right` (bỏ mất các lọ **bằng** ngưỡng)

In [28]:
def wrong_bisect_right(spells, potions, success):
    P = sorted(potions); m = len(P)
    return [m - bisect_right(P, (success + v - 1) // v) for v in spells]   # SAI: loại lọ đúng bằng need

for sp, po, s, _ in examples:
    print("bản sai :", wrong_bisect_right(sp, po, s), "| bản đúng:", Solution().successfulPairs(sp, po, s))

bản sai : [3, 0, 2] | bản đúng: [4, 0, 3]
bản sai : [2, 0, 0] | bản đúng: [2, 0, 2]


**Vì sao sai:** `bisect_right` đếm phần tử `> need`, nhưng điều kiện là `≥`. Ví dụ 2, bùa 2: `need = 8`, hai lọ `8` bị loại.

### Lỗi 3 — Quên sắp xếp `potions` trước khi tìm nhị phân

In [29]:
def wrong_unsorted(spells, potions, success):
    m = len(potions)
    return [m - bisect_left(potions, (success + v - 1) // v) for v in spells]   # SAI: bisect cần mảng đã sắp

for sp, po, s, _ in examples:
    print("bản sai :", wrong_unsorted(sp, po, s), "| bản đúng:", Solution().successfulPairs(sp, po, s))

bản sai : [4, 0, 3] | bản đúng: [4, 0, 3]
bản sai : [1, 0, 1] | bản đúng: [2, 0, 2]


**Vì sao sai:** ví dụ 1 có `potions` đã sắp sẵn nên **trùng hợp** đúng. Ví dụ 2: `[8, 5, 8]` chưa sắp — `bisect` cho kết quả vô nghĩa. **Sửa:** `sorted(potions)`.

### Lỗi 4 — Điều kiện **lớn hơn ngặt** (`> success`)

In [30]:
def wrong_strict(spells, potions, success):
    return [sum(1 for p in potions if v * p > success) for v in spells]     # SAI: đề yêu cầu >=

for sp, po, s, _ in examples:
    print("bản sai :", wrong_strict(sp, po, s), "| bản đúng:", Solution().successfulPairs(sp, po, s))

bản sai : [4, 0, 3] | bản đúng: [4, 0, 3]
bản sai : [2, 0, 0] | bản đúng: [2, 0, 2]


**Vì sao sai:** đề dùng "**at least** success" (≥). Ví dụ 1 **trùng hợp** đúng vì không có cặp nào có tích **đúng bằng** 7. Ví dụ 2: bùa 2 với hai lọ 8 cho tích đúng bằng 16 ⇒ bản sai đếm 0 thay vì 2. Bài học: đọc kỹ "at least / at most / strictly" và luôn thêm test có tích đúng bằng `success`.

### Lỗi 5 — Dùng float cho ngưỡng khi giá trị **lớn** (biến thể giới hạn tới 10¹⁸)

In [31]:
def float_threshold(spells, potions, success):
    P = sorted(potions); m = len(P)
    return [m - bisect_left(P, success / v) for v in spells]

rng = random.Random(5); found = None
for _ in range(20000):
    v = rng.randint(1, 10**3); p = rng.randint(10**15, 10**16); s = v * p
    if float_threshold([v], [p - 1, p], s) != Solution().successfulPairs([v], [p - 1, p], s):
        found = (v, p, s); break
v, p, s = found
print("v =", v, "p =", p, "success =", s)
print("bản sai (float):", float_threshold([v], [p - 1, p], s), "| bản đúng:", Solution().successfulPairs([v], [p - 1, p], s))
print("success / v =", repr(s / v), "nhưng giá trị thật là", p)

v = 287 p = 9253703859241649 success = 2655813007602353263
bản sai (float): [2] | bản đúng: [1]
success / v = 9253703859241648.0 nhưng giá trị thật là 9253703859241649


**Vì sao sai:** với số cỡ `10¹⁶`, float chỉ có ~16 chữ số có nghĩa: `s / v` bị làm tròn sang một số nguyên **khác** `p`. Ở giới hạn của đề (ngưỡng ≤ 10⁵) điều này không xảy ra, nhưng thói quen dùng số nguyên giúp code đúng khi giới hạn thay đổi. **Sửa:** trần số nguyên, hoặc so sánh bằng phép nhân như cách hai con trỏ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Với mỗi phần tử A, đếm số phần tử B thỏa điều kiện **đơn điệu** theo B" ⇒ **sắp xếp B + tìm nhị phân**.
- Điều kiện tích / tổng ⇒ đổi thành ngưỡng: `v·p ≥ s ⇔ p ≥ ⌈s/v⌉`; `a + b ≥ s ⇔ b ≥ s − a`.
- Trần nguyên: `(a + b − 1) // b` hoặc `-(-a // b)`; tránh float khi số lớn.
- `≥ t` ⇒ `bisect_left`; `> t` ⇒ `bisect_right`.
- Offline (biết hết truy vấn) ⇒ sắp xếp cả truy vấn + hai con trỏ; giá trị nhỏ ⇒ đếm + tổng hậu tố.
- Bài cùng họ: 2563 Count the Number of Fair Pairs, 1855 Maximum Distance Between a Pair of Values, 881 Boats to Save People, 1498 Number of Subsequences That Satisfy the Given Sum Condition, 2389 Longest Subsequence With Limited Sum.

## 15. Biến thể của bài

### 15.1 Biến thể A — Tổng số cặp thành công (một số), và số cặp có tích trong khoảng `[lo, hi]`

In [32]:
def count_in_range(spells, potions, lo, hi):
    P = sorted(potions); out = 0
    for v in spells:
        a = (lo + v - 1) // v                    # p >= ceil(lo / v)
        b = hi // v                              # p <= floor(hi / v)
        if a <= b: out += bisect_right(P, b) - bisect_left(P, a)
    return out

rng = random.Random(151)
for _ in range(1000):
    sp = [rng.randint(1, 20) for _ in range(rng.randint(1, 6))]; po = [rng.randint(1, 20) for _ in range(rng.randint(1, 6))]
    lo = rng.randint(1, 200); hi = lo + rng.randint(0, 200)
    assert count_in_range(sp, po, lo, hi) == sum(1 for v in sp for p in po if lo <= v * p <= hi)
print(sum(Solution().successfulPairs(*examples[0][:3])), count_in_range([5, 1, 3], [1, 2, 3, 4, 5], 7, 15))

7 5


### 15.2 Biến thể B — Giá trị tới 10¹⁸ (dùng số nguyên Python, không float)

In [33]:
def big_values(spells, potions, success):
    return Solution().successfulPairs(spells, potions, success)        # bản nộp đã dùng số nguyên

rng = random.Random(152)
for _ in range(500):
    sp = [rng.randint(1, 10**9) for _ in range(5)]; po = [rng.randint(1, 10**18) for _ in range(6)]
    s = rng.randint(1, 10**27)
    assert big_values(sp, po, s) == brute(sp, po, s)
print("số nguyên lớn OK")

số nguyên lớn OK


### 15.3 Biến thể C — Truy vấn trực tuyến: thêm lọ thuốc mới xen kẽ truy vấn (`SortedList`)

In [34]:
from sortedcontainers import SortedList

class Lab:
    def __init__(self, success):
        self.s = success; self.P = SortedList()
    def add_potion(self, p):
        self.P.add(p)
    def query(self, v):
        return len(self.P) - self.P.bisect_left((self.s + v - 1) // v)

lab = Lab(7)
for p in [1, 2]: lab.add_potion(p)
print(lab.query(5), end=" "); lab.add_potion(4); print(lab.query(5), lab.query(3), lab.query(1))

1 2 1 0


### 15.4 Biến thể D — Với mỗi bùa, lọ **yếu nhất** vẫn thành công (hoặc −1)

In [35]:
def weakest_success(spells, potions, success):
    P = sorted(potions); out = []
    for v in spells:
        k = bisect_left(P, (success + v - 1) // v)
        out.append(P[k] if k < len(P) else -1)
    return out

print(weakest_success(*examples[0][:3]), weakest_success(*examples[1][:3]))

[2, -1, 3] [8, -1, 8]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | định nghĩa | O(nm) | O(1) | kiểm chứng |
| 2. `bisect` + float | doocs | O((n+m) log m) | O(m) | ngắn, đúng trong giới hạn đề |
| 3. `bisect` + trần nguyên | ngưỡng chính xác | O((n+m) log m) | O(m) | **mặc định** |
| 4. Hai con trỏ | offline, chỉ nhân | O(n log n + m log m) | O(n+m) | tránh chia hoàn toàn |
| 5. Đếm + hậu tố | giá trị nhỏ | O(n + m + V) | O(V) | V nhỏ, rất nhiều truy vấn |
| 6. Nhớ theo giá trị | bùa trùng | O(m log m + u log m) | O(m + u) | nhiều bùa lặp |
| 7. Đảo vai | mảng hiệu trên bùa | O(n log n + m log n) | O(n) | m ≪ n |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `successfulPairs([1, 2, 3], [5, 5, 5], 10)` và `successfulPairs([6], [1, 1, 2], 12)`?

**Bài 2 (tính tay):** `spells = [4, 7]`, `potions = [3, 1, 6, 2, 5]`, `success = 20`. Tính `need` và đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(a, b)` trả về `⌈a / b⌉` cho `a ≥ 0, b > 0` **không** dùng phép chia số thực, và **không** dùng `+ b − 1`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(spells, potions, success)` cài cách **hai con trỏ** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(spells, potions, success)` trả về số cặp `(i, j)` với `spells[i] + potions[j] ≥ success` cho **mỗi** `i` (đổi tích thành tổng), O((n + m) log m).

In [36]:
def bt3_todo(a, b):
    # TODO: trần của a / b
    pass

def bt4_todo(spells, potions, success):
    # TODO: hai con trỏ
    pass

def bt5_todo(spells, potions, success):
    # TODO: điều kiện tổng
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `need = 10, 5, 4`. Mức 2: bùa 1 cần lọ ≥ 10 — không có. Mức 3: `[0, 3, 3]`; câu hai: `need = 2` ⇒ `[1]`.

**Bài 2.** Mức 1: sắp xếp `[1, 2, 3, 5, 6]`. Mức 2: `need = ⌈20/4⌉ = 5`, `⌈20/7⌉ = 3`. Mức 3: `[2, 3]`.

**Bài 3.** Mức 1: `−(−a // b)`. Mức 2: hoặc `q, r = divmod(a, b); q + (r > 0)`. Mức 3: chứng minh: `//` làm tròn xuống, nên `−(−a // b)` làm tròn lên.

**Bài 4.** Mức 1: sắp bùa theo giá trị, giữ chỉ số. Mức 2: con trỏ `k = m`. Mức 3: `while k > 0 and v * P[k − 1] >= success: k −= 1`.

**Bài 5.** Mức 1: `v + p ≥ s ⇔ p ≥ s − v`. Mức 2: `need = s − v` (có thể ≤ 0 ⇒ mọi lọ). Mức 3: `m − bisect_left(P, need)` đúng cả khi `need ≤ 0`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[0, 3, 3]`, `[1]`. **Bài 2:** `[2, 3]`.

In [37]:
assert Solution().successfulPairs([1, 2, 3], [5, 5, 5], 10) == [0, 3, 3]
assert Solution().successfulPairs([6], [1, 1, 2], 12) == [1]
assert Solution().successfulPairs([4, 7], [3, 1, 6, 2, 5], 20) == [2, 3]

def bt3(a, b):
    return -(-a // b)

def bt4(spells, potions, success):
    return two_pointers(spells, potions, success)

def bt5(spells, potions, success):
    P = sorted(potions); m = len(P)
    return [m - bisect_left(P, success - v) for v in spells]

rng = random.Random(1992)
for _ in range(2000):
    a = rng.randint(0, 10**12); b = rng.randint(1, 10**6)
    assert bt3(a, b) == (a + b - 1) // b
    sp = [rng.randint(1, 30) for _ in range(rng.randint(1, 6))]; po = [rng.randint(1, 30) for _ in range(rng.randint(1, 6))]
    s = rng.randint(1, 60)
    assert bt4(sp, po, s * 10) == brute(sp, po, s * 10)
    assert bt5(sp, po, s) == [sum(1 for p in po if v + p >= s) for v in sp]
print(bt3(7, 5), bt3(10, 5), bt3(0, 3), bt5([1, 10], [1, 2, 3], 4))
print("Đáp án bài tập: tất cả assert OK")

2 2 0 [1, 3]
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
P = sorted(potions); m = len(P)
return [m - bisect_left(P, (success + v - 1) // v) for v in spells]
#        số lọ >= ceil(success / v)   <=>   v * p >= success
```

**Checklist:**
- [ ] Đã sắp xếp `potions`.
- [ ] Ngưỡng là **trần** `⌈success / v⌉`, tính bằng số nguyên.
- [ ] `≥` ⇒ `bisect_left`.
- [ ] Ngưỡng có thể lớn hơn mọi lọ (đáp án 0) — `bisect_left` tự xử lý.